In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
spark = SparkSession.builder.appName("modelling").config("spark.driver.memory", "12g").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 17:41:08 WARN Utils: Your hostname, aydin-khan-desktop, resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlp11s0)
26/08/29 17:41:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/29 17:41:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where app

In [2]:
yellow_taxi_data = spark.read.parquet('cleaned_data/yellow_data.parquet')
yellow_taxi_data.show(1)

+-------------------+------------+------------+-------+
|               date|PULocationID|total_amount|Borough|
+-------------------+------------+------------+-------+
|2023-05-01 00:42:49|         138|       57.15| Queens|
+-------------------+------------+------------+-------+
only showing top 1 row


In [3]:
from utils import add_stations
# aggregate to daily
yellow_taxi_daily_earnings = (
    yellow_taxi_data
    .withColumn("date", F.to_date("date"))
    .groupBy("date", "PULocationID", 'Borough')
    .agg(F.sum("total_amount").alias("daily_total_amount"))
    .orderBy("date")
)
yellow_taxi_daily_earnings = add_stations(yellow_taxi_daily_earnings, spark)

residential_data = spark.read.csv('cleaned_data/residential_data.csv', header=True, inferSchema=True)
residential_data = residential_data.withColumn("residential_density", F.col("TotalResidentialUnits") / F.col("TotalNumBldgs"))
residential_data = residential_data.select('LocationID', 'residential_density')
training_data = yellow_taxi_daily_earnings.join(residential_data, yellow_taxi_daily_earnings.PULocationID == residential_data.LocationID, 'left').drop('LocationID')

# drop rows where residential_density is null
print('Shape', training_data.count(), ',', len(training_data.columns))
training_data.show(2)

/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_versi

Shape 262332 , 6


+----------+------------+---------+------------------+------------+-------------------+
|      date|PULocationID|  Borough|daily_total_amount|num_stations|residential_density|
+----------+------------+---------+------------------+------------+-------------------+
|2023-05-01|         165| Brooklyn|            128.86|           3| 2.4132334581772783|
|2023-05-02|          79|Manhattan| 36476.00999999998|           2| 16.148442272449604|
+----------+------------+---------+------------------+------------+-------------------+
only showing top 2 rows


In [4]:
# remove all areas that have NULL residential_density except with PULocationID 1, 132 or 138
training_data = training_data.filter(~((F.col('residential_density').isNull()) & (~F.col('PULocationID').isin([1, 132, 138]))))
# create a new column called isAirport that is 1 if PULocationID is 132, 1 or 138, else 0
training_data = training_data.withColumn('isAirport', F.when(F.col('PULocationID').isin([132, 1, 138]), 1).otherwise(0))

# replace null values in residential_density with 0
training_data = training_data.fillna({'residential_density': 0})
print('Shape', training_data.count(), ',', len(training_data.columns))

Shape 252380 , 7


In [5]:
fhvhv_taxi_data = spark.read.parquet('cleaned_data/fhvhv_data.parquet')
fhvhv_taxi_data = fhvhv_taxi_data.withColumn("date", F.to_date("date"))
fhvhv_trip_volume = (
    fhvhv_taxi_data
    .groupBy("date", "PULocationID")
    .agg(F.count("*").alias("daily_trip_volume"))
    .orderBy("date")
)
fhvhv_taxi_data.show()

+----------+------------+
|      date|PULocationID|
+----------+------------+
|2023-03-01|         114|
|2023-03-01|          79|
|2023-03-01|         113|
|2023-03-01|         229|
|2023-03-01|         144|
|2023-03-01|           7|
|2023-03-01|         255|
|2023-03-01|         255|
|2023-03-01|          80|
|2023-03-01|         255|
|2023-03-01|          80|
|2023-03-01|         255|
|2023-03-01|         231|
|2023-03-01|         229|
|2023-03-01|          36|
|2023-03-01|         112|
|2023-03-01|         141|
|2023-03-01|         161|
|2023-03-01|          75|
|2023-03-01|         139|
+----------+------------+
only showing top 20 rows


In [6]:
training_data = training_data.join(
    fhvhv_trip_volume,
    on=["date", "PULocationID"],
    how='inner'
)
training_data.show()

+----------+------------+---------+------------------+------------+-------------------+---------+-----------------+
|      date|PULocationID|  Borough|daily_total_amount|num_stations|residential_density|isAirport|daily_trip_volume|
+----------+------------+---------+------------------+------------+-------------------+---------+-----------------+
|2023-01-01|          11| Brooklyn|              42.6|           0| 2.4930684874818954|        0|              703|
|2023-01-01|          24|Manhattan|3790.2100000000014|           1|  25.82941176470588|        0|              906|
|2023-01-01|          91| Brooklyn|              54.0|           0| 1.4156811499509965|        0|             3037|
|2023-01-01|         123| Brooklyn|             43.65|           1|  2.507860262008734|        0|             1721|
|2023-01-01|         137|Manhattan| 20404.77999999995|           0| 35.746736292428196|        0|             3458|
|2023-01-01|         174|    Bronx|              35.4|           3|  16.